In [3]:
import os
from pathlib import Path

# Move from 'notebooks/' to 'src/' to access local modules
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent / "src")

print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: D:\TUSZ_project\src


In [4]:
import re
import shutil
from tqdm.auto import tqdm
import mne
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tusz_datalake_manager import (
    create_metadata, 
    clean_bi_files, 
    restructure_directory
)

RAW_DATA_PATH = Path(r"D:\TUSZ_project\TUSZ_DataLake\01_Raw_Consolidated")
METADATA_PATH = Path(r"D:\TUSZ_project\TUSZ_DataLake\02_Metadata")
RANDOM_SEED = 42
TRAIN_TEST_RATIO = 0.7

# Metadata

## REVISAR PACIENTE 50 y primera sesion del 209

In [3]:
df_p, df_s = create_metadata(RAW_DATA_PATH, METADATA_PATH,seed = RANDOM_SEED, train_ratio = TRAIN_TEST_RATIO)

Scanning TRAIN:   0%|          | 0/4667 [00:00<?, ?it/s]

Scanning DEV:   0%|          | 0/1832 [00:00<?, ?it/s]

Scanning EVAL:   0%|          | 0/865 [00:00<?, ?it/s]


TUSZ DATALAKE AUDIT REPORT (Seed: 42 | Ratio: 0.70)
Total Partitions Scanned:      7364
Partitions Excluded (-1):      249
Partitions Retained (0 or 1):  7115
------------------------------------------------------------
Total Unique Patients Found:   675
Unique Patients Retained:      675
Total Usable Signal Duration:  1240.27 hours
------------------------------------------------------------
SPLIT DISTRIBUTION (70/30)
TRAIN (0): 472 patients | 827.60 hours
TEST  (1): 203 patients | 412.67 hours



In [4]:
display(df_p.head())

,patient_num_id,patient_id,original_split,age,gender,n_sessions,total_duration_sec,split_final
0,1,aaaaaaac,train,55,2,4,3015.0,0
1,2,aaaaaaag,train,38,2,3,4600.0,1
2,3,aaaaaaaq,eval,28,2,3,9175.0,0
3,4,aaaaaaar,train,89,2,1,1238.0,0
4,5,aaaaaaav,train,28,2,1,1992.0,1


In [5]:
display(df_s.head())

,patient_num_id,session_id,partition_id,config_type,file_name,sfreq,n_channels,duration_sec,highpass,lowpass,date,split_final
0,673,s001,t000,01_tcp_ar,aaaaatth_s001_t000.edf,1000.0,30,1289.0,0.0,500.0,2015-01-01,1
1,674,s001,t000,01_tcp_ar,aaaaatvk_s001_t000.edf,256.0,34,1290.0,0.0,128.0,2015-01-01,0
2,674,s002,t001,01_tcp_ar,aaaaatvk_s002_t001.edf,256.0,34,301.0,0.0,128.0,2015-01-01,0
3,674,s002,t002,01_tcp_ar,aaaaatvk_s002_t002.edf,256.0,34,301.0,0.0,128.0,2015-01-01,0
4,674,s002,t004,01_tcp_ar,aaaaatvk_s002_t004.edf,256.0,34,301.0,0.0,128.0,2015-01-01,0


In [6]:
df_p[df_p['patient_id'] == 'aaaaajrs']

,patient_num_id,patient_id,original_split,age,gender,n_sessions,total_duration_sec,split_final
334,335,aaaaajrs,train,61,1,1,1307.0,0


# Restructuración

In [7]:
print("Starting physical removal of redundant .csv_bi files...")
clean_bi_files(RAW_DATA_PATH)
remaining = list(Path(RAW_DATA_PATH).rglob("*.csv_bi"))
if not remaining:
    print("Verification successful: No .csv_bi files found in the Data Lake.")
else:
    print(f"Alert: {len(remaining)} files could not be deleted (likely in use by another process).")

Starting physical removal of redundant .csv_bi files...


Cleaning artifacts:   0%|          | 0/7364 [00:00<?, ?it/s]

Verification successful: No .csv_bi files found in the Data Lake.


In [8]:
print("Commencing final DataLake restructuring...")
restructure_directory(RAW_DATA_PATH, df_p)

Commencing final DataLake restructuring...


Flattening configs:   0%|          | 0/1643 [00:00<?, ?it/s]

Consolidating train:   0%|          | 0/579 [00:00<?, ?it/s]

Consolidating dev:   0%|          | 0/53 [00:00<?, ?it/s]

Consolidating eval:   0%|          | 0/43 [00:00<?, ?it/s]

Datalake physical restructuring completed.
